In [1]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 26.9 MB/s eta 0:00:00


In [2]:
import glob, os
import pandas as pd

ROOT = glob.glob("/kaggle/input/**/lingoqa", recursive=True)[0]
EVAL = f"{ROOT}/evaluation/evaluation"
df = pd.read_parquet(f"{ROOT}/evaluation.parquet")
print(len(os.listdir(EVAL)), "clips,", df["question_id"].nunique(), "questions")

100 clips, 500 questions


In [3]:
# import torch
# from transformers import AutoProcessor, AutoModelForMultimodalLM

# MODEL = "Qwen/Qwen3.5-2B"
# processor = AutoProcessor.from_pretrained(MODEL)
# model = AutoModelForMultimodalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")

In [4]:
# from PIL import Image

# def load_frames(seg):
#     imgs = []
#     for i in range(5):
#         img = Image.open(f"{EVAL}/{seg}/{i}.jpg").convert("RGB")
#         img.thumbnail((768, 768))
#         imgs.append(img)
#     return imgs

# def ask(seg, question):
#     content = [{"type": "image", "image": im} for im in load_frames(seg)]
#     content.append({"type": "text", "text": question + "\nAnswer in one short sentence."})
#     inputs = processor.apply_chat_template(
#         [{"role": "user", "content": content}],
#         add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
#     ).to(model.device)
#     out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
#     return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# row = df.iloc[0]
# print(row["question"], "\nTRUE:", row["answer"], "\nMODEL:", ask(row["segment_id"], row["question"]))

In [5]:
# import time, csv

# qs = df.drop_duplicates("question_id").reset_index(drop=True)
# OUT = "/kaggle/working/preds_qwen35_2b_A0.csv"
# done = set(pd.read_csv(OUT)["question_id"]) if os.path.exists(OUT) else set()

# N = 500
# t0 = time.time()
# with open(OUT, "a", newline="") as f:
#     w = csv.writer(f)
#     if not done:
#         w.writerow(["question_id", "segment_id", "answer"])
#     for i, r in qs.head(N).iterrows():
#         if r["question_id"] in done:
#             continue
#         w.writerow([r["question_id"], r["segment_id"], ask(r["segment_id"], r["question"])])
#         f.flush()
#         if i % 25 == 0:
#             print(i, f"{(time.time() - t0) / (i + 1):.1f}s per question", flush=True)

In [6]:
# from transformers import pipeline

# judge = pipeline("text-classification", model="wayveai/Lingo-Judge", device=0)
# preds = pd.read_csv(OUT).set_index("question_id")["answer"]

# def judge_score(r):
#     text = f"[CLS]\nQuestion: {r['question']}\nAnswer: {r['answer']}\nStudent: {preds[r['question_id']]}"
#     return judge(text)[0]["score"]

# ev = df[df["question_id"].isin(preds.index)].copy()
# ev["score"] = ev.apply(judge_score, axis=1)
# per_q = ev.groupby("question_id")["score"].max() > 0.5
# print(f"A0 Lingo-Judge: {100 * per_q.mean():.1f}%  on {len(per_q)} questions")

In [7]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [8]:
import gc, time, traceback, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForMultimodalLM

qs = df.drop_duplicates("question_id").reset_index(drop=True)

def load_frames(seg, blank=False):
    imgs = []
    for i in range(5):
        img = Image.open(f"{EVAL}/{seg}/{i}.jpg").convert("RGB")
        img.thumbnail((768, 768))
        imgs.append(Image.new("RGB", img.size) if blank else img)
    return imgs

def ask(model, processor, seg, question, blank=False):
    content = [{"type": "image", "image": im} for im in load_frames(seg, blank)]
    content.append({"type": "text", "text": question + "\nAnswer in one short sentence."})
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=model.dtype)
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [9]:
def run_model(model_id, tag, dtypes=(torch.float16,), n=500, blank=False):
    out_csv = f"/kaggle/working/preds_{tag}.csv"
    for dt in dtypes:
        model = processor = None
        try:
            processor = AutoProcessor.from_pretrained(model_id)
            model = AutoModelForMultimodalLM.from_pretrained(model_id, dtype=dt, device_map="cuda")
            first = ask(model, processor, qs["segment_id"][0], qs["question"][0], blank)
            print(tag, dt, "first answer:", repr(first), flush=True)
            if not first or set(first) <= set("!?. "):
                raise ValueError("empty or garbage output")
            t0, rows = time.time(), []
            for i, r in qs.head(n).iterrows():
                rows.append([r["question_id"], r["segment_id"],
                             ask(model, processor, r["segment_id"], r["question"], blank)])
                if i % 50 == 0:
                    print(tag, i, f"{(time.time() - t0) / (i + 1):.1f}s per question", flush=True)
            pd.DataFrame(rows, columns=["question_id", "segment_id", "answer"]).to_csv(out_csv, index=False)
            return out_csv
        except Exception:
            traceback.print_exc()
            print(tag, "FAILED with", dt, flush=True)
        finally:
            del model, processor
            gc.collect(); torch.cuda.empty_cache()
    return None

In [10]:
RUNS = [
    ("Qwen/Qwen3.5-2B",           "qwen35_2b",         dict()),
    ("Qwen/Qwen3.5-2B",           "qwen35_2b_blank20", dict(n=20, blank=True)),
    ("Qwen/Qwen3-VL-2B-Instruct", "qwen3vl_2b",        dict()),
    ("google/gemma-4-E2B-it",     "gemma4_e2b",        dict(dtypes=(torch.float16, torch.bfloat16))),
]
files = {tag: run_model(mid, tag, **kw) for mid, tag, kw in RUNS}
print(files)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (29 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

gemma4_e2b torch.float16 first answer: 'Yes, there is a traffic light visible in the distance, but its color is not clearly discernible.'
gemma4_e2b 0 2.7s per question
gemma4_e2b 50 2.6s per question
gemma4_e2b 100 2.5s per question
gemma4_e2b 150 2.5s per question
gemma4_e2b 200 2.5s per question
gemma4_e2b 250 2.6s per question
gemma4_e2b 300 2.6s per question
gemma4_e2b 350 2.6s per question
gemma4_e2b 400 2.6s per question
gemma4_e2b 450 2.6s per question
{'qwen35_2b': '/kaggle/working/preds_qwen35_2b.csv', 'qwen35_2b_blank20': '/kaggle/working/preds_qwen35_2b_blank20.csv', 'qwen3vl_2b': '/kaggle/working/preds_qwen3vl_2b.csv', 'gemma4_e2b': '/kaggle/working/preds_gemma4_e2b.csv'}


In [11]:
from transformers import pipeline
judge = pipeline("text-classification", model="wayveai/Lingo-Judge", device=0)

def score(csv_path, keep=None):
    p = pd.read_csv(csv_path).set_index("question_id")["answer"].fillna("").astype(str)
    if keep is not None:
        p = p[p.index.isin(keep)]
    ev = df[df["question_id"].isin(p.index)]
    texts = [f"[CLS]\nQuestion: {q}\nAnswer: {a.lower().strip()}\nStudent: {p[qid].lower().strip()}"
             for q, a, qid in zip(ev["question"], ev["answer"], ev["question_id"])]
    ev = ev.assign(score=[o["score"] for o in judge(texts, batch_size=32, truncation=True, max_length=128)])
    return round(100 * (ev.groupby("question_id")["score"].max() > 0.5).mean(), 1), len(p)

first20 = set(qs["question_id"][:20])
rows = [(tag, *score(path)) for tag, path in files.items() if path]
if files.get("qwen35_2b"):
    rows.append(("qwen35_2b_first20", *score(files["qwen35_2b"], keep=first20)))
res = pd.DataFrame(rows, columns=["run", "lingo_judge", "questions"])
res.to_csv("/kaggle/working/bakeoff.csv", index=False)
print(res.to_string(index=False))

config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  738MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

              run  lingo_judge  questions
        qwen35_2b         65.4        500
qwen35_2b_blank20         30.0         20
       qwen3vl_2b         49.8        500
       gemma4_e2b         29.2        500
qwen35_2b_first20         60.0         20
